# 🛡️ GUARDIAN AI — SECURED System
### ✅ Information Security Concepts Applied

| Concept | Applied |
|---|---|
| 🔐 Authentication | JWT Tokens + bcrypt Password Hashing |
| 👮 Authorization | Role-Based Access Control (RBAC) |
| 🔒 Encryption | bcrypt (passwords) + AES Fernet (case data) |
| 📋 Audit Logs | Every action logged (user, IP, timestamp) |
| 🛡️ Input Validation | SQL Injection + XSS Protection |
| ⏱️ Rate Limiting | Brute Force Protection (15min lockout) |
| 🚫 CSRF Protection | Token-based CSRF prevention |
| 👥 User Management | Admin can add/delete/reset Inspector logins |

---
### 👥 All Login Credentials:
| Username | Password | Role | Area Access |
|---|---|---|---|
| admin_alishba | admin123 | Admin | ALL areas + user management |
| sho_faisalabad | sho456 | SHO | Hollywood only |
| public_user | user789 | User | Chatbot only |
| insp_bilal | bilal123 | Inspector | Hollywood cases |
| insp_sana | sana123 | Inspector | Central cases |
| insp_raza | raza123 | Inspector | Newton cases |
| insp_ayesha | ayesha123 | Inspector | Southwest cases |
| insp_tariq | tariq123 | Inspector | Rampart cases |
| insp_ahmed | ahmed123 | Inspector | 77Th Street cases |

---
### 🔑 Admin User Management Routes:
| Route | Action |
|---|---|
| GET /users | All users list dekhna |
| POST /users/add | Naya inspector/user banana |
| POST /users/delete/<id> | User delete karna |
| POST /users/reset_password/<id> | Password reset karna |
| POST /users/unlock/<id> | Locked account unlock karna |

## 📦 CELL 1 — Install Libraries (Security Libraries Added)

In [1]:
# ✅ CELL 1: Install all required libraries including security ones
!pip install flask flask-cors scikit-learn pandas numpy joblib --quiet

# 🔐 Security libraries
!pip install bcrypt pyjwt cryptography flask-limiter bleach --quiet

print('\n✅ All libraries installed!')
print('🔐 Security libraries: bcrypt, PyJWT, cryptography, flask-limiter, bleach')

ERROR: Could not find a version that satisfies the requirement xgboost (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for xgboost



✅ All libraries installed!
🔐 Security libraries: bcrypt, PyJWT, cryptography, flask-limiter, bleach



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔐 CELL 2 — Security Configuration

In [2]:
# ✅ CELL 2: Security config — JWT secret, AES key, rate limit settings
import os
import secrets
import base64
from cryptography.fernet import Fernet

# ─── 🔑 SECRET KEYS (auto-generated — not hardcoded!) ───────────────────────
# These are generated fresh every time server starts
# In production: store these in environment variables

FLASK_SECRET_KEY = secrets.token_hex(32)          # 64-char random hex
JWT_SECRET_KEY   = secrets.token_hex(32)          # For JWT tokens
AES_KEY          = Fernet.generate_key()          # For AES encryption
fernet           = Fernet(AES_KEY)                # Encryption object

# ─── ⏱️ RATE LIMITING CONFIG ────────────────────────────────────────────────
MAX_LOGIN_ATTEMPTS = 5       # 5 se zyada galat login → account lock
LOCKOUT_MINUTES    = 15      # 15 minute ke liye lock
JWT_EXPIRY_HOURS   = 2       # JWT token 2 ghante baad expire

# ─── 🛡️ ALLOWED ROLES ──────────────────────────────────────────────────────
VALID_ROLES = ['Admin', 'SHO', 'User']

print('✅ Security configuration loaded!')
print(f'🔑 Flask Secret Key: {FLASK_SECRET_KEY[:8]}... (auto-generated)')
print(f'🎫 JWT Secret Key:   {JWT_SECRET_KEY[:8]}... (auto-generated)')
print(f'🔒 AES Key:          {AES_KEY[:8]}... (auto-generated)')
print(f'⏱️  Max Login Attempts: {MAX_LOGIN_ATTEMPTS}')
print(f'⏱️  Lockout Duration:   {LOCKOUT_MINUTES} minutes')
print(f'🎫 JWT Expiry:         {JWT_EXPIRY_HOURS} hours')

✅ Security configuration loaded!
🔑 Flask Secret Key: 46858001... (auto-generated)
🎫 JWT Secret Key:   1c537a41... (auto-generated)
🔒 AES Key:          b'hMgRyenE'... (auto-generated)
⏱️  Max Login Attempts: 5
⏱️  Lockout Duration:   15 minutes
🎫 JWT Expiry:         2 hours


## 🗄️ CELL 3 — Setup Secured Database

In [3]:
# ✅ CELL 3: Secured database with hashed passwords + extra security columns
import sqlite3
import bcrypt
from datetime import datetime

# ─── 🔒 PASSWORD HASHING FUNCTIONS ─────────────────────────────────────────
def hash_password(plain_password):
    """bcrypt se password hash karo — plain text kabhi store nahi hoga"""
    salt = bcrypt.gensalt(rounds=12)  # 12 rounds = strong hashing
    return bcrypt.hashpw(plain_password.encode('utf-8'), salt).decode('utf-8')

def verify_password(plain_password, hashed_password):
    """Login ke waqt password verify karo"""
    return bcrypt.checkpw(
        plain_password.encode('utf-8'),
        hashed_password.encode('utf-8')
    )

# ─── 🔒 AES ENCRYPTION FUNCTIONS ───────────────────────────────────────────
def encrypt_data(plain_text):
    """Sensitive case data ko AES se encrypt karo"""
    if not plain_text:
        return plain_text
    return fernet.encrypt(plain_text.encode()).decode()

def decrypt_data(encrypted_text):
    """Encrypted data wapas read karo"""
    if not encrypted_text:
        return encrypted_text
    try:
        return fernet.decrypt(encrypted_text.encode()).decode()
    except Exception:
        return '[ENCRYPTED — Cannot Decrypt]'

# ─── DATABASE SETUP ─────────────────────────────────────────────────────────
def init_secured_db():
    conn = sqlite3.connect('CriminalInvestigation_Secured.db')
    cursor = conn.cursor()

    # Users table — password column ab HASH store karega, plain text nahi
    # login_attempts + locked_until = brute force protection
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Users (
            u_id            INTEGER PRIMARY KEY AUTOINCREMENT,
            username        TEXT UNIQUE NOT NULL,
            password_hash   TEXT NOT NULL,
            role            TEXT NOT NULL CHECK(role IN ("Admin","SHO","User","Inspector")),
            assigned_area   TEXT,
            login_attempts  INTEGER DEFAULT 0,
            locked_until    DATETIME,
            last_login      DATETIME,
            created_at      DATETIME DEFAULT CURRENT_TIMESTAMP
        )''')

    # Cases table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Criminal_Cases (
            case_id           INTEGER PRIMARY KEY AUTOINCREMENT,
            case_title        TEXT NOT NULL,
            crime_type        TEXT NOT NULL,
            area              TEXT NOT NULL,
            status            TEXT DEFAULT "Open" CHECK(status IN ("Open","Closed","Under Investigation")),
            sho_id            INTEGER,
            investigator_name TEXT,
            created_at        DATETIME DEFAULT CURRENT_TIMESTAMP
        )''')

    # Case_Details — full_story aur evidence AES encrypted store hogi
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Case_Details (
            detail_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            case_id        INTEGER,
            full_story     TEXT,
            suspect_info   TEXT,
            evidence_notes TEXT,
            is_encrypted   INTEGER DEFAULT 1
        )''')

    # Audit Logs — ab ip_address aur status bhi track hoga
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Logs (
            log_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id       INTEGER,
            username      TEXT,
            action        TEXT NOT NULL,
            table_affected TEXT,
            ip_address    TEXT,
            status        TEXT DEFAULT "SUCCESS",
            timestamp     DATETIME DEFAULT CURRENT_TIMESTAMP
        )''')

    # CSRF Tokens table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS CSRF_Tokens (
            token_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id    INTEGER,
            token      TEXT UNIQUE,
            expires_at DATETIME,
            used       INTEGER DEFAULT 0
        )''')

    # ─── Insert users with HASHED passwords ─────────────────────────────────
    print('⏳ Hashing passwords with bcrypt (rounds=12)...')
    users = [
        # Admin
        ('admin_alishba',   hash_password('admin123'),  'Admin',      'All'),
        # SHO
        ('sho_faisalabad',  hash_password('sho456'),    'SHO',        'Hollywood'),
        # Public
        ('public_user',     hash_password('user789'),   'User',       'None'),
        # Inspectors — each assigned to their area
        ('insp_bilal',      hash_password('bilal123'),  'Inspector',  'Hollywood'),
        ('insp_sana',       hash_password('sana123'),   'Inspector',  'Central'),
        ('insp_raza',       hash_password('raza123'),   'Inspector',  'Newton'),
        ('insp_ayesha',     hash_password('ayesha123'), 'Inspector',  'Southwest'),
        ('insp_tariq',      hash_password('tariq123'),  'Inspector',  'Rampart'),
        ('insp_ahmed',      hash_password('ahmed123'),  'Inspector',  '77Th Street'),
    ]
    cursor.executemany(
        'INSERT OR IGNORE INTO Users (username, password_hash, role, assigned_area) VALUES (?,?,?,?)',
        users
    )

    # ─── Insert sample cases ─────────────────────────────────────────────────
    cases = [
        ('Bank Robbery',      'Theft',      'Hollywood',   'Open',   2, 'Inspector Bilal'),
        ('Cyber Fraud',       'Cyber',      'Central',     'Open',   2, 'Inspector Sana'),
        ('Assault Case',      'Violence',   'Newton',      'Closed', 2, 'Inspector Raza'),
        ('Kidnapping Report', 'Violent',    'Southwest',   'Open',   2, 'Inspector Ayesha'),
        ('Drug Bust',         'Narcotics',  'Rampart',     'Open',   2, 'Inspector Tariq'),
        ('Vehicle Theft Ring','Theft',      '77Th Street', 'Open',   2, 'Inspector Ahmed'),
        ('Burglary Series',   'Burglary',   'Pacific',     'Open',   2, 'Inspector Bilal'),
        ('Battery Incident',  'Assault',    'Olympic',     'Closed', 2, 'Inspector Sana'),
    ]
    cursor.executemany(
        'INSERT OR IGNORE INTO Criminal_Cases (case_title,crime_type,area,status,sho_id,investigator_name) VALUES (?,?,?,?,?,?)',
        cases
    )

    conn.commit()
    conn.close()
    print('✅ Secured Database ready!')
    print('🔒 Passwords stored as bcrypt hashes — NOT plain text')
    print('🔐 Case details will be AES encrypted')
    print('📋 Audit logs track: user, action, IP, timestamp, status')
    print('👮 Users created: Admin(1), SHO(1), Inspector(6), Public(1)')
    print('\n👥 Inspector Logins:')
    print('   insp_bilal  / bilal123   → Hollywood')
    print('   insp_sana   / sana123    → Central')
    print('   insp_raza   / raza123    → Newton')
    print('   insp_ayesha / ayesha123  → Southwest')
    print('   insp_tariq  / tariq123   → Rampart')
    print('   insp_ahmed  / ahmed123   → 77Th Street')

init_secured_db()

⏳ Hashing passwords with bcrypt (rounds=12)...
✅ Secured Database ready!
🔒 Passwords stored as bcrypt hashes — NOT plain text
🔐 Case details will be AES encrypted
📋 Audit logs track: user, action, IP, timestamp, status
👮 Users created: Admin(1), SHO(1), Inspector(6), Public(1)

👥 Inspector Logins:
   insp_bilal  / bilal123   → Hollywood
   insp_sana   / sana123    → Central
   insp_raza   / raza123    → Newton
   insp_ayesha / ayesha123  → Southwest
   insp_tariq  / tariq123   → Rampart
   insp_ahmed  / ahmed123   → 77Th Street


## 🤖 CELL 4 — Train XGBoost Model (Same as before)

In [ ]:
# ✅ CELL 4: Random Forest Crime Prediction Model (train or load artifacts)
# Does NOT modify security routes — only supplies model, le, area_crime, top_crimes for /chat_predict
import sys
import subprocess
from pathlib import Path

ROOT = Path('.').resolve()
ML_DIR = ROOT / 'ml'
ARTIFACTS = ML_DIR / 'artifacts'
sys.path.insert(0, str(ML_DIR))

DATA_CANDIDATES = [
    ROOT / 'Professional_Cleaned_Crime_Data.csv',
    Path('/content/Professional_Cleaned_Crime_Data.csv'),
    Path('/content/sample_data/Professional_Cleaned_Crime_Data.csv'),
]
DATA_CSV = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_CSV is None:
    raise FileNotFoundError('Professional_Cleaned_Crime_Data.csv not found. Place it in the project root.')

MODEL_FILE = ARTIFACTS / 'crime_rf_model.joblib'
if not MODEL_FILE.exists():
    print('⏳ No saved model found — training Random Forest Classifier...')
    print(f'   Dataset: {DATA_CSV}')
    subprocess.check_call([
        sys.executable, str(ML_DIR / 'train_model.py'),
        '--data', str(DATA_CSV),
        '--output', str(ARTIFACTS),
        '--n-estimators', '120',
    ])
else:
    print('✅ Found trained model artifacts — loading from disk')

from chat_predict_service import load_chat_artifacts

_chat = load_chat_artifacts(ARTIFACTS)
rf_model = _chat['rf_model']
encoders = _chat['encoders']
aggregates = _chat['aggregates']
le = _chat['le']
area_crime = _chat['area_crime']
top_crimes = _chat['top_crimes']

print('✅ Random Forest crime prediction model ready')
print(f'   Areas supported: {len(le.classes_)}')
print(f'   Crime types in model: {len(_chat["crime_classes"])}')
print('   Target: 1 = Crime Likely | 0 = Crime Not Likely')

✅ Found trained model artifacts — loading from disk
✅ Random Forest crime prediction model ready
   Areas supported: 21
   Crime types in model: 140
   Target: 1 = Crime Likely | 0 = Crime Not Likely


## 🌐 CELL 5 — Secured Flask App (All 7 Security Concepts)

In [5]:
# ✅ CELL 5: SECURED Flask backend — all 7 InfoSec concepts applied
# ─── Fix: ML path + load artifacts if Cell 4 was not run ───────────────────
import sys
from pathlib import Path as _Path
_PROJECT_ROOT = _Path('.').resolve()
_ML_DIR = _PROJECT_ROOT / 'ml'
if str(_ML_DIR) not in sys.path:
    sys.path.insert(0, str(_ML_DIR))

def _ensure_ml_loaded():
    global rf_model, encoders, aggregates, le, area_crime, top_crimes
    try:
        rf_model
    except NameError:
        from chat_predict_service import load_chat_artifacts
        _c = load_chat_artifacts(_ML_DIR / 'artifacts')
        rf_model = _c['rf_model']
        encoders = _c['encoders']
        aggregates = _c['aggregates']
        le = _c['le']
        area_crime = _c['area_crime']
        top_crimes = _c['top_crimes']
        print('✅ ML artifacts auto-loaded for Cell 5')

_ensure_ml_loaded()

from flask import Flask, request, jsonify, session
from flask_cors import CORS
from flask_limiter import Limiter
from flask_limiter.util import get_remote_address
import sqlite3
import jwt
import bleach
import secrets
import re
from datetime import datetime, timedelta
from functools import wraps

app = Flask(__name__)
app.secret_key = FLASK_SECRET_KEY   # Auto-generated — not hardcoded!
app.config['PERMANENT_SESSION_LIFETIME'] = timedelta(hours=JWT_EXPIRY_HOURS)

# ─── 🚫 CORS — sirf apna frontend allow karo, '*' nahi ──────────────────────
CORS(app,
     supports_credentials=True,
     origins=['http://localhost:5173',   # Vite dev server (primary)
              'http://localhost:3000',   # CRA fallback
              'https://*.ngrok-free.app', # ngrok
              'https://*.ngrok.io']        # ngrok alternate
)

# ─── 🌐 CORS PREFLIGHT SAFETY NET ───────────────────────────────────────────
# This handles OPTIONS preflight requests that Flask-CORS sometimes misses
@app.after_request
def add_cors_headers(response):
    allowed_origins = [
        'http://localhost:5173',
        'http://localhost:3000',
    ]
    origin = request.headers.get('Origin', '')
    if origin in allowed_origins or 'ngrok' in origin:
        response.headers['Access-Control-Allow-Origin']      = origin
        response.headers['Access-Control-Allow-Credentials'] = 'true'
        response.headers['Access-Control-Allow-Methods']     = 'GET, POST, PUT, DELETE, OPTIONS'
        response.headers['Access-Control-Allow-Headers']     = 'Content-Type, Authorization, X-CSRF-Token, ngrok-skip-browser-warning'
    return response

@app.route('/', defaults={'path': ''}, methods=['OPTIONS'])
@app.route('/<path:path>', methods=['OPTIONS'])
def handle_options(path):
    """Handle all OPTIONS preflight requests"""
    response = app.make_default_options_response()
    origin = request.headers.get('Origin', '')
    if origin and ('localhost' in origin or 'ngrok' in origin):
        response.headers['Access-Control-Allow-Origin']      = origin
        response.headers['Access-Control-Allow-Credentials'] = 'true'
        response.headers['Access-Control-Allow-Methods']     = 'GET, POST, PUT, DELETE, OPTIONS'
        response.headers['Access-Control-Allow-Headers']     = 'Content-Type, Authorization, X-CSRF-Token, ngrok-skip-browser-warning'
    return response

# ─── ⏱️ RATE LIMITING — Brute Force Protection ──────────────────────────────
limiter = Limiter(
    get_remote_address,
    app=app,
    default_limits=['200 per day', '50 per hour'],
    storage_uri='memory://'
)

# ─── DB HELPER ───────────────────────────────────────────────────────────────
def get_db():
    conn = sqlite3.connect('CriminalInvestigation_Secured.db')
    conn.row_factory = sqlite3.Row
    return conn

# ─── 📋 AUDIT LOG ───────────────────────────────────────────────────────────
def log_action(user_id, username, action, table, status='SUCCESS'):
    """Har action log hoga — kon, kab, kya, kahan se, success ya fail"""
    ip = request.remote_addr or 'unknown'
    conn = get_db()
    conn.execute(
        'INSERT INTO Logs (user_id, username, action, table_affected, ip_address, status) VALUES (?,?,?,?,?,?)',
        (user_id, username, action, table, ip, status)
    )
    conn.commit()
    conn.close()

# ─── 🛡️ INPUT VALIDATION — XSS + SQL Injection Protection ──────────────────
def sanitize_input(text, max_length=200):
    """User input se HTML tags aur special characters hataao"""
    if not text or not isinstance(text, str):
        return ''
    # bleach se XSS tags strip karo
    clean = bleach.clean(str(text), tags=[], strip=True)
    # Length limit
    clean = clean[:max_length]
    # SQL injection ke common patterns hataao
    sql_patterns = ["'--", '; DROP', 'UNION SELECT', '1=1', 'OR 1', 'xp_']
    for pattern in sql_patterns:
        clean = clean.replace(pattern, '')
    return clean.strip()

def validate_username(username):
    """Username sirf letters, numbers, underscore allow karo"""
    if not username or len(username) > 50:
        return False
    return bool(re.match(r'^[a-zA-Z0-9_]+$', username))

# ─── 🔐 JWT TOKEN FUNCTIONS ─────────────────────────────────────────────────
def generate_jwt(user_id, username, role, area):
    """Login pe JWT token generate karo"""
    payload = {
        'user_id':  user_id,
        'username': username,
        'role':     role,
        'area':     area,
        'exp':      datetime.utcnow() + timedelta(hours=JWT_EXPIRY_HOURS),  # Expiry
        'iat':      datetime.utcnow(),   # Issued at
        'jti':      secrets.token_hex(16) # Unique token ID
    }
    return jwt.encode(payload, JWT_SECRET_KEY, algorithm='HS256')

def verify_jwt(token):
    """Token verify karo — expired ya tampered tokens reject"""
    try:
        return jwt.decode(token, JWT_SECRET_KEY, algorithms=['HS256'])
    except jwt.ExpiredSignatureError:
        return None   # Token expire ho gaya
    except jwt.InvalidTokenError:
        return None   # Token tampered hai

# ─── 👮 RBAC DECORATOR — Role check karo ────────────────────────────────────
def require_auth(allowed_roles=None):
    """Har protected route pe role check karo"""
    def decorator(f):
        @wraps(f)
        def wrapper(*args, **kwargs):
            # JWT token header se lo
            token = request.headers.get('Authorization', '').replace('Bearer ', '')
            if not token:
                # Fallback: session se check karo
                if not session.get('user_id'):
                    log_action(0, 'anonymous', f'Unauthorized access: {request.path}', 'Auth', 'FAILED')
                    return jsonify({'status': 'denied', 'message': 'Login required'}), 401
                payload = {
                    'user_id':  session.get('user_id'),
                    'username': session.get('username'),
                    'role':     session.get('role'),
                    'area':     session.get('area')
                }
            else:
                payload = verify_jwt(token)
                if not payload:
                    log_action(0, 'unknown', 'Invalid/Expired JWT token', 'Auth', 'FAILED')
                    return jsonify({'status': 'denied', 'message': 'Token expired or invalid'}), 401

            # Role check
            if allowed_roles and payload.get('role') not in allowed_roles:
                log_action(payload.get('user_id'), payload.get('username'),
                           f'Forbidden: {request.path}', 'Auth', 'FORBIDDEN')
                return jsonify({'status': 'denied', 'message': 'Access forbidden'}), 403

            request.current_user = payload
            return f(*args, **kwargs)
        return wrapper
    return decorator

# ─── 🚫 CSRF TOKEN FUNCTIONS ────────────────────────────────────────────────
def generate_csrf_token(user_id):
    """State-changing requests ke liye CSRF token"""
    token = secrets.token_hex(32)
    expires = datetime.utcnow() + timedelta(hours=1)
    conn = get_db()
    conn.execute(
        'INSERT INTO CSRF_Tokens (user_id, token, expires_at) VALUES (?,?,?)',
        (user_id, token, expires)
    )
    conn.commit()
    conn.close()
    return token

def verify_csrf_token(token, user_id):
    """CSRF token valid hai ya nahi check karo"""
    conn = get_db()
    row = conn.execute(
        "SELECT * FROM CSRF_Tokens WHERE token=? AND user_id=? AND used=0 AND expires_at > datetime('now')",
        (token, user_id)
    ).fetchone()
    if row:
        conn.execute('UPDATE CSRF_Tokens SET used=1 WHERE token=?', (token,))
        conn.commit()
    conn.close()
    return row is not None

# ════════════════════════════════════════════════════════════════════════════
# ─── ROUTES ─────────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════════════

# ─── 🔐 LOGIN — Rate limited + bcrypt verify ────────────────────────────────
@app.route('/login', methods=['POST'])
@limiter.limit('10 per minute')   # ⏱️ Sirf 10 login attempts per minute
def login():
    data = request.json or {}

    # 🛡️ Input Validation
    username = sanitize_input(data.get('username', ''))
    password = data.get('password', '')

    if not validate_username(username):
        log_action(0, username, 'Login failed: invalid username format', 'Users', 'FAILED')
        return jsonify({'status': 'fail', 'message': 'Invalid credentials'}), 401

    if not password or len(password) > 100:
        return jsonify({'status': 'fail', 'message': 'Invalid credentials'}), 401

    conn = get_db()
    user = conn.execute(
        'SELECT * FROM Users WHERE username=?', (username,)
    ).fetchone()

    # ⏱️ Account locked check
    if user and user['locked_until']:
        lock_time = datetime.strptime(user['locked_until'], '%Y-%m-%d %H:%M:%S.%f') \
                    if '.' in str(user['locked_until']) \
                    else datetime.strptime(str(user['locked_until']), '%Y-%m-%d %H:%M:%S')
        if datetime.utcnow() < lock_time:
            minutes_left = int((lock_time - datetime.utcnow()).seconds / 60) + 1
            log_action(user['u_id'], username, 'Login blocked: account locked', 'Users', 'BLOCKED')
            conn.close()
            return jsonify({'status': 'fail',
                            'message': f'Account locked. Try after {minutes_left} minutes'}), 429

    # 🔒 bcrypt password verify
    if user and verify_password(password, user['password_hash']):
        # Reset attempts on successful login
        conn.execute(
            'UPDATE Users SET login_attempts=0, locked_until=NULL, last_login=? WHERE u_id=?',
            (datetime.utcnow(), user['u_id'])
        )
        conn.commit()
        conn.close()

        # 🔐 Generate JWT
        token = generate_jwt(user['u_id'], user['username'], user['role'], user['assigned_area'])

        # Session bhi set karo (for cookie-based auth)
        session.permanent = True
        session['user_id']  = user['u_id']
        session['username'] = user['username']
        session['role']     = user['role']
        session['area']     = user['assigned_area']

        # 🚫 Generate CSRF token
        csrf_token = generate_csrf_token(user['u_id'])

        log_action(user['u_id'], username, f'Login successful', 'Users', 'SUCCESS')

        return jsonify({
            'status':     'success',
            'role':       user['role'],
            'area':       user['assigned_area'],
            'username':   user['username'],
            'token':      token,         # JWT token frontend ko do
            'csrf_token': csrf_token     # CSRF token bhi do
        })

    else:
        # ⏱️ Failed attempt count badhaao
        if user:
            new_attempts = user['login_attempts'] + 1
            locked_until = None
            if new_attempts >= MAX_LOGIN_ATTEMPTS:
                locked_until = datetime.utcnow() + timedelta(minutes=LOCKOUT_MINUTES)
                print(f'🔒 Account {username} locked until {locked_until}')
            conn.execute(
                'UPDATE Users SET login_attempts=?, locked_until=? WHERE u_id=?',
                (new_attempts, locked_until, user['u_id'])
            )
            conn.commit()
        conn.close()

        log_action(0, username, 'Login failed: wrong credentials', 'Users', 'FAILED')
        return jsonify({'status': 'fail', 'message': 'Invalid credentials'}), 401

@app.route('/logout', methods=['GET','POST'])
def logout():
    username = session.get('username', 'unknown')
    user_id  = session.get('user_id', 0)
    session.clear()
    log_action(user_id, username, 'Logout', 'Users', 'SUCCESS')
    return jsonify({'status': 'success'})

# ─── 👮 CASES — RBAC applied ─────────────────────────────────────────────────
@app.route('/cases', methods=['GET'])
@require_auth(allowed_roles=['Admin', 'SHO', 'Inspector'])
def get_cases():
    user = request.current_user
    conn = get_db()
    if user['role'] == 'Admin':
        rows = conn.execute('SELECT * FROM Criminal_Cases ORDER BY case_id DESC').fetchall()
    elif user['role'] in ['SHO', 'Inspector']:
        # 👮 SHO aur Inspector — sirf apna area dekhein
        rows = conn.execute(
            'SELECT * FROM Criminal_Cases WHERE area=? ORDER BY case_id DESC',
            (user['area'],)
        ).fetchall()
    conn.close()
    log_action(user['user_id'], user['username'], 'Viewed cases', 'Criminal_Cases')
    return jsonify({'status': 'ok', 'cases': [dict(r) for r in rows]})

@app.route('/cases/add', methods=['POST'])
@require_auth(allowed_roles=['Admin', 'SHO', 'Inspector'])
def add_case():
    user = request.current_user
    d    = request.json or {}

    # 🚫 CSRF check
    csrf = d.get('csrf_token', '')
    if not verify_csrf_token(csrf, user['user_id']):
        log_action(user['user_id'], user['username'], 'CSRF token invalid on add_case', 'Criminal_Cases', 'BLOCKED')
        return jsonify({'status': 'denied', 'message': 'Invalid CSRF token'}), 403

    # 🛡️ Input sanitization
    case_title        = sanitize_input(d.get('case_title', 'Untitled'), 100)
    crime_type        = sanitize_input(d.get('crime_type', 'Unknown'), 50)
    area              = sanitize_input(d.get('area', ''), 50)
    investigator_name = sanitize_input(d.get('investigator_name', 'Unknown'), 100)

    if not case_title or not area:
        return jsonify({'status': 'fail', 'message': 'case_title and area are required'}), 400

    conn = get_db()
    cur = conn.execute(
        'INSERT INTO Criminal_Cases (case_title,crime_type,area,status,sho_id,investigator_name) VALUES (?,?,?,?,?,?)',
        (case_title, crime_type, area, 'Open', user['user_id'], investigator_name)
    )
    new_id = cur.lastrowid
    conn.commit()
    conn.close()
    log_action(user['user_id'], user['username'], f'Added Case #{new_id}: {case_title}', 'Criminal_Cases')
    return jsonify({'status': 'success', 'case_id': new_id})

@app.route('/cases/update/<int:case_id>', methods=['POST'])
@require_auth(allowed_roles=['Admin', 'SHO', 'Inspector'])
def update_case(case_id):
    user = request.current_user
    d    = request.json or {}

    # 🚫 CSRF check
    if not verify_csrf_token(d.get('csrf_token', ''), user['user_id']):
        return jsonify({'status': 'denied', 'message': 'Invalid CSRF token'}), 403

    # 🛡️ Validate status value
    allowed_statuses = ['Open', 'Closed', 'Under Investigation']
    new_status = d.get('status', 'Open')
    if new_status not in allowed_statuses:
        return jsonify({'status': 'fail', 'message': 'Invalid status value'}), 400

    investigator = sanitize_input(d.get('investigator_name', ''), 100)

    conn = get_db()
    conn.execute(
        'UPDATE Criminal_Cases SET status=?, investigator_name=? WHERE case_id=?',
        (new_status, investigator, case_id)
    )
    conn.commit()
    conn.close()
    log_action(user['user_id'], user['username'], f'Updated Case #{case_id} → {new_status}', 'Criminal_Cases')
    return jsonify({'status': 'success'})

@app.route('/cases/delete/<int:case_id>', methods=['POST'])
@require_auth(allowed_roles=['Admin'])   # 👮 Sirf Admin delete kar sakta hai
def delete_case(case_id):
    user = request.current_user
    d    = request.json or {}

    # 🚫 CSRF check
    if not verify_csrf_token(d.get('csrf_token', ''), user['user_id']):
        return jsonify({'status': 'denied', 'message': 'Invalid CSRF token'}), 403

    conn = get_db()
    conn.execute('DELETE FROM Case_Details WHERE case_id=?', (case_id,))
    conn.execute('DELETE FROM Criminal_Cases WHERE case_id=?', (case_id,))
    conn.commit()
    conn.close()
    log_action(user['user_id'], user['username'], f'DELETED Case #{case_id}', 'Criminal_Cases')
    return jsonify({'status': 'success'})

# ─── 🔒 ENCRYPTED CASE DETAILS ──────────────────────────────────────────────
@app.route('/cases/<int:case_id>/details', methods=['POST'])
@require_auth(allowed_roles=['Admin', 'SHO', 'Inspector'])
def save_case_details(case_id):
    """Sensitive case info AES se encrypt karke save karo"""
    user = request.current_user
    d    = request.json or {}

    # 🔒 AES encrypt karo sensitive fields
    encrypted_story    = encrypt_data(sanitize_input(d.get('full_story',''), 2000))
    encrypted_suspect  = encrypt_data(sanitize_input(d.get('suspect_info',''), 1000))
    encrypted_evidence = encrypt_data(sanitize_input(d.get('evidence_notes',''), 1000))

    conn = get_db()
    conn.execute(
        'INSERT OR REPLACE INTO Case_Details (case_id, full_story, suspect_info, evidence_notes, is_encrypted) VALUES (?,?,?,?,1)',
        (case_id, encrypted_story, encrypted_suspect, encrypted_evidence)
    )
    conn.commit()
    conn.close()
    log_action(user['user_id'], user['username'], f'Saved encrypted details for Case #{case_id}', 'Case_Details')
    return jsonify({'status': 'success', 'message': 'Details saved with AES encryption'})

@app.route('/cases/<int:case_id>/details', methods=['GET'])
@require_auth(allowed_roles=['Admin', 'SHO', 'Inspector'])
def get_case_details(case_id):
    """Encrypted case details decrypt karke return karo"""
    user = request.current_user
    conn = get_db()
    row  = conn.execute('SELECT * FROM Case_Details WHERE case_id=?', (case_id,)).fetchone()
    conn.close()
    if not row:
        return jsonify({'status': 'not_found'}), 404
    log_action(user['user_id'], user['username'], f'Viewed details for Case #{case_id}', 'Case_Details')
    return jsonify({
        'status':         'ok',
        'full_story':     decrypt_data(row['full_story']),
        'suspect_info':   decrypt_data(row['suspect_info']),
        'evidence_notes': decrypt_data(row['evidence_notes'])
    })

# ─── 🤖 AI CHATBOT (Random Forest — full feature prediction) ────────────────
from chat_predict_service import process_chat_message

@app.route('/chat_predict', methods=['POST'])
@limiter.limit('30 per minute')   # ⏱️ Chatbot ke liye bhi rate limit
def chat_predict():
    data = request.json or {}

    # 🛡️ Input validation — message sanitize karo
    raw_message = data.get('message', '')
    user_query  = sanitize_input(raw_message, max_length=500)

    def _fetch_live_case(area_name):
        conn = get_db()
        row = conn.execute(
            "SELECT case_title, investigator_name FROM Criminal_Cases "
            "WHERE area=? AND status='Open' LIMIT 1",
            (area_name,),
        ).fetchone()
        conn.close()
        if row:
            return {
                'case_title': row['case_title'],
                'investigator_name': row['investigator_name'],
            }
        return None

    result = process_chat_message(
        user_query,
        rf_model,
        encoders,
        aggregates,
        get_live_case=_fetch_live_case,
    )
    return jsonify(result)

# ─── 🗺️ HEATMAP + ANALYTICS APIs (extension — no schema change) ─────────────
from guardian_api_extensions import register_extension_routes
register_extension_routes(app, sanitize_input, limiter)

# ─── 📋 AUDIT LOGS — Admin only ──────────────────────────────────────────────
@app.route('/logs', methods=['GET'])
@require_auth(allowed_roles=['Admin'])
def get_logs():
    conn = get_db()
    rows = conn.execute(
        'SELECT * FROM Logs ORDER BY log_id DESC LIMIT 100'
    ).fetchall()
    conn.close()
    return jsonify({'logs': [dict(r) for r in rows]})

# ─── 🎫 CSRF Token endpoint ─────────────────────────────────────────────────
@app.route('/csrf_token', methods=['GET'])
@require_auth()
def get_csrf_token():
    user  = request.current_user
    token = generate_csrf_token(user['user_id'])
    return jsonify({'csrf_token': token})


# ─── 👥 USER MANAGEMENT — Admin Only ────────────────────────────────────────

@app.route('/users', methods=['GET'])
@require_auth(allowed_roles=['Admin'])
def get_users():
    """Admin — sab users dekhe"""
    user = request.current_user
    conn = get_db()
    rows = conn.execute(
        'SELECT u_id, username, role, assigned_area, login_attempts, last_login, created_at FROM Users ORDER BY u_id'
    ).fetchall()
    conn.close()
    log_action(user['user_id'], user['username'], 'Viewed all users', 'Users')
    return jsonify({'status': 'ok', 'users': [dict(r) for r in rows]})

@app.route('/users/add', methods=['POST'])
@require_auth(allowed_roles=['Admin'])
def add_user():
    """Admin — naya user/inspector add kare"""
    user = request.current_user
    d    = request.json or {}

    username  = sanitize_input(d.get('username', ''), 50)
    password  = d.get('password', '')
    role      = d.get('role', 'Inspector')
    area      = sanitize_input(d.get('assigned_area', ''), 50)

    if not username or not password:
        return jsonify({'status': 'fail', 'message': 'Username and password required'}), 400

    if role not in ['Admin', 'SHO', 'Inspector', 'User']:
        return jsonify({'status': 'fail', 'message': 'Invalid role'}), 400

    if len(password) < 6:
        return jsonify({'status': 'fail', 'message': 'Password must be at least 6 characters'}), 400

    hashed = hash_password(password)
    conn   = get_db()
    try:
        conn.execute(
            'INSERT INTO Users (username, password_hash, role, assigned_area) VALUES (?,?,?,?)',
            (username, hashed, role, area)
        )
        conn.commit()
        conn.close()
        log_action(user['user_id'], user['username'], f'Added user: {username} ({role})', 'Users')
        return jsonify({'status': 'success', 'message': f'User {username} added successfully!'})
    except Exception as e:
        conn.close()
        return jsonify({'status': 'fail', 'message': 'Username already exists'}), 400

@app.route('/users/delete/<int:uid>', methods=['POST'])
@require_auth(allowed_roles=['Admin'])
def delete_user(uid):
    """Admin — user delete kare"""
    user = request.current_user
    if uid == user['user_id']:
        return jsonify({'status': 'fail', 'message': 'Cannot delete your own account'}), 400
    conn = get_db()
    conn.execute('DELETE FROM Users WHERE u_id=?', (uid,))
    conn.commit()
    conn.close()
    log_action(user['user_id'], user['username'], f'Deleted user #{uid}', 'Users')
    return jsonify({'status': 'success'})

@app.route('/users/reset_password/<int:uid>', methods=['POST'])
@require_auth(allowed_roles=['Admin'])
def reset_password(uid):
    """Admin — kisi bhi user ka password reset kare"""
    user     = request.current_user
    d        = request.json or {}
    new_pass = d.get('new_password', '')

    if len(new_pass) < 6:
        return jsonify({'status': 'fail', 'message': 'Password must be at least 6 characters'}), 400

    hashed = hash_password(new_pass)
    conn   = get_db()
    conn.execute(
        'UPDATE Users SET password_hash=?, login_attempts=0, locked_until=NULL WHERE u_id=?',
        (hashed, uid)
    )
    conn.commit()
    conn.close()
    log_action(user['user_id'], user['username'], f'Reset password for user #{uid}', 'Users')
    return jsonify({'status': 'success', 'message': 'Password reset successfully!'})

@app.route('/users/unlock/<int:uid>', methods=['POST'])
@require_auth(allowed_roles=['Admin'])
def unlock_user(uid):
    """Admin — locked account unlock kare"""
    user = request.current_user
    conn = get_db()
    conn.execute(
        'UPDATE Users SET login_attempts=0, locked_until=NULL WHERE u_id=?', (uid,)
    )
    conn.commit()
    conn.close()
    log_action(user['user_id'], user['username'], f'Unlocked user #{uid}', 'Users')
    return jsonify({'status': 'success', 'message': 'Account unlocked!'})

# ─── HEALTH CHECK ───────────────────────────────────────────────────────────
@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'security': 'enabled', 'model': 'RandomForest'})

print('✅ SECURED Flask app built!')
print('🔐 JWT Authentication:   ON')
print('👮 RBAC Authorization:   ON (Admin/SHO/Inspector/User)')
print('🔒 bcrypt + AES:         ON')
print('📋 Audit Logging:        ON (IP + status tracked)')
print('🛡️  Input Validation:     ON (XSS + SQL injection)')
print('⏱️  Rate Limiting:        ON (10/min login, 30/min chat)')
print('🚫 CSRF Protection:      ON')
print('👥 User Management:      ON (Admin can add/delete/reset inspector logins)')
print('\n👥 Inspector Access:')
print('   insp_bilal  → Hollywood cases')
print('   insp_sana   → Central cases')
print('   insp_raza   → Newton cases')
print('   insp_ayesha → Southwest cases')
print('   insp_tariq  → Rampart cases')
print('   insp_ahmed  → 77Th Street cases')

✅ SECURED Flask app built!
🔐 JWT Authentication:   ON
👮 RBAC Authorization:   ON (Admin/SHO/Inspector/User)
🔒 bcrypt + AES:         ON
📋 Audit Logging:        ON (IP + status tracked)
🛡️  Input Validation:     ON (XSS + SQL injection)
⏱️  Rate Limiting:        ON (10/min login, 30/min chat)
🚫 CSRF Protection:      ON
👥 User Management:      ON (Admin can add/delete/reset inspector logins)

👥 Inspector Access:
   insp_bilal  → Hollywood cases
   insp_sana   → Central cases
   insp_raza   → Newton cases
   insp_ayesha → Southwest cases
   insp_tariq  → Rampart cases
   insp_ahmed  → 77Th Street cases


## 🚀 CELL 6 — Launch Secured Server

In [6]:
# ✅ CELL 6: Start secured Flask (+ optional ngrok)
import threading, time, os
try:
    from pyngrok import ngrok
    HAS_NGROK = True
except ImportError:
    HAS_NGROK = False
    print('ℹ️ pyngrok not installed — using localhost only (pip install pyngrok for tunnel)')

try:
    os.system('fuser -k 5000/tcp 2>/dev/null')
except Exception:
    pass
time.sleep(1)

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(3)

try:
    if not HAS_NGROK:
        raise ImportError('pyngrok not available')
    public_url = ngrok.connect(5000).public_url
    print('\n' + '='*60)
    print('🛡️  GUARDIAN AI SECURED IS LIVE!')
    print('='*60)
    print(f'\n🔗 YOUR SECURED API URL:\n   {public_url}')
    print(f'\n📋 Test Links:')
    print(f'   Health : {public_url}/health')
    print(f'   Login  : POST {public_url}/login')
    print(f'\n🎯 Paste this URL in GuardianAI_Secured.jsx:')
    print(f'   const FLASK_BASE = "{public_url}";')
    print('\n🔐 Security Features Active:')
    print('   ✅ bcrypt password hashing')
    print('   ✅ JWT tokens (2hr expiry)')
    print('   ✅ Rate limiting (10 login/min)')
    print('   ✅ CSRF protection')
    print('   ✅ AES encrypted case details')
    print('   ✅ Full audit logging with IP')
    print('='*60)
except Exception as e:
    from google.colab.output import eval_js
    colab_url = eval_js('google.colab.kernel.proxyPort(5000)')
    print(f'\n🔗 Colab URL: {colab_url}')
    print(f'Error with ngrok: {e}')

ModuleNotFoundError: No module named 'pyngrok'

## 🧪 CELL 7 — Test Security Features

In [ ]:
# ✅ CELL 7: Test all security features
import requests, time
BASE = 'http://localhost:5000'

print('\n--- 🧪 SECURITY TESTS ---\n')

# Test 1: Normal login
s = requests.Session()
r = s.post(f'{BASE}/login', json={'username':'admin_alishba','password':'admin123'})
data = r.json()
print(f'✅ Test 1 - Normal Login: {data.get("status")} | Role: {data.get("role")}')
jwt_token  = data.get('token')
csrf_token = data.get('csrf_token')
print(f'   JWT Token: {jwt_token[:30]}...')
print(f'   CSRF Token: {csrf_token[:20]}...')

# Test 2: Wrong password
r2 = requests.post(f'{BASE}/login', json={'username':'admin_alishba','password':'wrongpass'})
print(f'\n✅ Test 2 - Wrong Password: {r2.json().get("message")} (HTTP {r2.status_code})')

# Test 3: SQL injection attempt
r3 = requests.post(f'{BASE}/login', json={'username':"admin' OR '1'='1", 'password':'anything'})
print(f'\n✅ Test 3 - SQL Injection: {r3.json().get("message")} (HTTP {r3.status_code})')

# Test 4: XSS attempt in chat
r4 = s.post(f'{BASE}/chat_predict', json={'message':'<script>alert(1)</script> Central'})
reply = r4.json().get('reply','')
has_script = '<script>' in reply
print(f'\n✅ Test 4 - XSS in Chat: Script tag blocked = {not has_script}')

# Test 5: RBAC — public user cannot see cases
s2 = requests.Session()
s2.post(f'{BASE}/login', json={'username':'public_user','password':'user789'})
r5 = s2.get(f'{BASE}/cases')
print(f'\n✅ Test 5 - RBAC: Public user cases access = {r5.json().get("status")} (HTTP {r5.status_code})')

# Test 6: Brute force — 6 wrong attempts
print(f'\n⏱️  Test 6 - Brute Force Protection:')
test_s = requests.Session()
for i in range(6):
    r6 = test_s.post(f'{BASE}/login', json={'username':'sho_faisalabad','password':f'wrong{i}'})
    status_msg = r6.json().get('message','')
    if 'locked' in status_msg.lower():
        print(f'   Attempt {i+1}: 🔒 ACCOUNT LOCKED — {status_msg}')
        break
    print(f'   Attempt {i+1}: {status_msg} (HTTP {r6.status_code})')

# Test 7: Audit logs
r7 = s.get(f'{BASE}/logs')
logs = r7.json().get('logs', [])
print(f'\n📋 Test 7 - Audit Logs ({len(logs)} entries):')
for log in logs[:5]:
    print(f'   [{log["timestamp"]}] {log["username"]} | {log["action"]} | {log["status"]} | IP: {log["ip_address"]}')

print('\n✅ All security tests complete!')